# 03 — S3 Tables (Apache Iceberg)
**Objective:** native Iceberg tables on ObjectScale via Spark's REST catalog — no data movement.

**Customer sees:** table bucket + namespace, DDL/DML, MERGE, schema evolution, time travel, and a read-only denial.
**TC:** TC3.1–TC3.7

## Step 1 — Session
SparkSession with the `os` REST catalog pointed at the OBS TableHead endpoint (SigV4, table bucket ARN prefix).

In [ ]:
import pandas as pd
from IPython.display import HTML, display
from poc import config, results, viz, s3util
import poc.spark_helpers as sh
spark = sh.build_session("obs-poc")
viz.table(__import__("pandas").DataFrame(sh.versions(spark).items(), columns=["component","version"]))
print("users:", config.masked_users())
print("Spark UI:", config.SPARK_UI)

**Presenter notes** (hidden)

Catalog os -> http://192.168.1.31:19020/iceberg, warehouse=poc-lakehouse ARN.

## Step 2 — Table bucket & namespace via S3 Tables API (TC3.1)
Create/confirm the table bucket and namespace through the AWS-compatible API and show the responses.

In [ ]:
st = s3util.s3tables("tables-rw")
arn = f"arn:aws:s3tables:::bucket/{config.OBS_TABLE_BUCKET}"
ns = config.TABLES_NAMESPACE
tbl = st.list_table_buckets()["tableBuckets"]
namespaces = [n for n in st.list_namespaces(tableBucketARN=arn).get("namespaces",[])]
viz.table(pd.DataFrame(tbl)); viz.table(pd.DataFrame(namespaces, columns=["namespace"]))
res = results.write("TC3.1", "bucket+namespace", bool(tbl) and bool(namespaces))
viz.banner(res["passed"], "TC3.1", f"bucket={config.OBS_TABLE_BUCKET} ns={ns}")

**Presenter notes** (hidden)

boto3 s3tables client + x-emc-namespace header.

## Step 3 — SHOW NAMESPACES + CREATE TABLE (TC3.2)
The same objects are now visible through Spark's Iceberg catalog — one control plane, no ETL.

In [ ]:
spark.sql("SHOW NAMESPACES IN os").show()
spark.sql(f"DROP TABLE IF EXISTS os.{config.TABLES_NAMESPACE}.orders_ice")
spark.sql(f"""CREATE TABLE os.{config.TABLES_NAMESPACE}.orders_ice
 (order_id string, customer string, status string, amount double, qty int)
 USING iceberg""")
res = results.write("TC3.2", "create table", True)
viz.banner(res["passed"], "TC3.2", "os.pocns.orders_ice created")

**Presenter notes** (hidden)

SparkCatalog 'os' — type=rest, SigV4-signed.

## Step 4 — INSERT + SELECT (TC3.3)
Insert rows through the Iceberg catalog; they land in the managed `--table-s3` bucket on ObjectScale.

In [ ]:
spark.sql(f"""INSERT INTO os.{config.TABLES_NAMESPACE}.orders_ice
 VALUES ('o1','cust-001','PAID',199.0,2),('o2','cust-002','NEW',49.5,1),
        ('o3','cust-003','SHIPPED',1200.0,10)""")
rows = spark.sql(f"SELECT * FROM os.{config.TABLES_NAMESPACE}.orders_ice").toPandas()
viz.table(rows)
res = results.write("TC3.3", "insert+select", len(rows) == 3, {"rows": len(rows)})
viz.banner(res["passed"], "TC3.3")

**Presenter notes** (hidden)

Data files visible in OBS UI under <uuid>--table-s3.

## Step 5 — MERGE INTO (TC3.4)
Upsert a changed row — Iceberg's row-level update over object storage, before and after side by side.

In [ ]:
before = rows.copy()
spark.sql(f"""MERGE INTO os.{config.TABLES_NAMESPACE}.orders_ice t
 USING (SELECT 'o2' order_id,'cust-002' customer,'PAID' status,55.0 amount,1 qty) u
 ON t.order_id=u.order_id WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
after = spark.sql(f"SELECT * FROM os.{config.TABLES_NAMESPACE}.orders_ice ORDER BY order_id").toPandas()
viz.table(before.assign(state="before")); viz.table(after.assign(state="after"))
res = results.write("TC3.4", "merge", after.loc[after.order_id=="o2","status"].iloc[0]=="PAID")
viz.banner(res["passed"], "TC3.4", "o2 NEW -> PAID")

**Presenter notes** (hidden)

MERGE INTO needs the Iceberg SQL extensions — already configured.

## Step 6 — Schema evolution (TC3.5)
Add a column without rewriting data.

In [ ]:
spark.sql(f"ALTER TABLE os.{config.TABLES_NAMESPACE}.orders_ice ADD COLUMNS (region string)")
cols = spark.sql(f"DESCRIBE os.{config.TABLES_NAMESPACE}.orders_ice").toPandas()
viz.table(cols)
res = results.write("TC3.5", "add column", "region" in set(cols.col_name))
viz.banner(res["passed"], "TC3.5")

**Presenter notes** (hidden)

Metadata-only operation — instant.

## Step 7 — Time travel & snapshot history (TC3.6)
Query an earlier snapshot vs current, and show the snapshot log.

In [ ]:
snaps = spark.sql(f"SELECT snapshot_id, committed_at FROM os.{config.TABLES_NAMESPACE}.orders_ice.snapshots").toPandas()
old = spark.sql(f"SELECT count(*) n FROM os.{config.TABLES_NAMESPACE}.orders_ice VERSION AS OF {snaps.snapshot_id.iloc[0]}").toPandas()
cur = spark.sql(f"SELECT count(*) n FROM os.{config.TABLES_NAMESPACE}.orders_ice").toPandas()
viz.table(snaps); viz.table(pd.concat([old.assign(view="first snapshot"), cur.assign(view="current")]))
ok = len(snaps) >= 2 and int(old.n.iloc[0]) <= int(cur.n.iloc[0])
res = results.write("TC3.6", "time travel", ok,
                    {"snapshots": len(snaps), "first": int(old.n.iloc[0]),
                     "current": int(cur.n.iloc[0])})
viz.banner(res["passed"], "TC3.6", f"{len(snaps)} snapshots")

**Presenter notes** (hidden)

VERSION AS OF uses snapshot_id; TIMESTAMP AS OF also supported.

## Step 8 — Read-only denial (TC3.7)
Rebuild the session as tables-ro and run INSERT — should fail with Access Denied.

In [ ]:
spark.stop()
spark = sh.build_session("obs-poc-ro", tables_user="tables-ro")
try:
    spark.sql(f"INSERT INTO os.{config.TABLES_NAMESPACE}.orders_ice BY NAME SELECT 'x' AS order_id")
    denied, code = False, "unexpected success"
except Exception as e:
    denied, code = "Denied" in str(e) or "403" in str(e), str(e)[:60]
res = results.write("TC3.7", "ro denied", denied, {"error": code})
viz.banner(res["passed"], "TC3.7", f"Access Denied as expected")

**Presenter notes** (hidden)

tables-ro: S3TablesReadOnlyAccess + ECSS3ReadOnlyAccess.

## Step 9 — List tables via API
Back to the control plane: list tables in the namespace via S3 Tables API.

In [ ]:
tabs = st.list_tables(tableBucketARN=arn, namespace=config.TABLES_NAMESPACE)["tables"]
viz.table(pd.DataFrame(tabs)[["name","type"]])
viz.banner(True, "tables listed", f"{len(tabs)} table(s)")
spark.stop()

**Presenter notes** (hidden)

Same table list is visible in the OBS UI under S3 Tables.